In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

cwd = Path.cwd()
print(cwd)
root=cwd/'institutional-roi-analysis'
pd.set_option("display.max_columns",None)
display(root)

C:\Users\sebas\PycharmProjects\Git\Seb_branch


WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [4]:
tdf = pd.read_parquet(root/"data"/"raw"/"scorecard"/"national_scorecard.parquet")
display(tdf.head())
display(tdf.info())

,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,school.state,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type,latest.admissions.sat_scores.average.overall,latest.admissions.act_scores.midpoint.cumulative,latest.student.grad_students
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              55930 non-null  object 
 1   title                                             55930 non-null  object 
 2   unit_id                                           55930 non-null  int64  
 3   distance                                          55930 non-null  int64  
 4   school.type                                       55930 non-null  object 
 5   credential.level                                  55930 non-null  int64  
 6   earnings.1_yr.overall_median_earnings             46006 non-null  float64
 7   earnings.1_yr.working_not_enrolled.overall_count  46006 non-null  float64
 8   earnings.4_yr.overall_median_earnings             55930 non-null  int64  
 9   earnings.4_yr.wor

None

# Why so many missing Admission Rates?

In [5]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.3301

In [6]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.005

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [7]:
df = df.drop(columns="id")
df = clean(df)

C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df.columns = df.columns.str.replace(r".", "_")


Numeric columns: Index(['distance', 'credential_level', '1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count', 'location_lat',
       'location_lon', 'locale', 'carnegie_size_setting',
       'admission_rate_overall', 'median_family_income',
       'students_with_pell_grant', 'open_admissions_policy', 'age_entry',
       'title_iv_eligibility_type', 'sat_scores_average_overall',
       'act_scores_midpoint_cumulative', 'grad_students'],
      dtype='object')


In [8]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [9]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   code                            55930 non-null  string  
 1   title                           55930 non-null  string  
 2   unit_id                         55930 non-null  string  
 3   distance                        55930 non-null  int64   
 4   school_type                     55930 non-null  string  
 5   credential_level                55930 non-null  int64   
 6   1_yr_median_earnings            46006 non-null  float64 
 7   1_yr_working_count              46006 non-null  float64 
 8   4_yr_median_earnings            55930 non-null  int64   
 9   4_yr_working_count              55930 non-null  int64   
 10  5_yr_median_earnings            41343 non-null  float64 
 11  5_yr_working_count              41343 non-null  float64 
 12  school_name       

None

In [10]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,Alabama A & M University,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,Alabama A & M University,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,Alabama A & M University,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,Alabama A & M University,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,Alabama A & M University,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid


In [11]:
save(df,file_type="scorecard",clean=0,file_name="national_preprocessed_scorecard_programs")

In [12]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
count,55930.000000,55930.000000,46006.000000,46006.000000,55930.000000,55930.000000,41343.000000,41343.000000,55928.000000,12.000000,55930.000000,51620.000000,37469.000000,55566.000000,53574.000000,54911.000000,55566.000000,55930.000000,28424.000000,25623.000000,39457.000000
mean,1.298301,3.423368,49347.405447,110.133591,65571.535759,101.731164,63604.452483,108.088358,37.957356,145.037141,18.530753,11.347792,0.720972,42843.610319,0.633126,1.685819,23.105226,1.018309,1209.583697,25.467275,4507.224345
std,0.739933,5.726003,24051.517980,300.209665,27720.152721,280.310977,28396.672776,278.890877,5.427287,0.412824,9.133195,4.841543,0.231800,22801.900981,0.176964,0.464193,3.379662,0.228855,143.918529,4.015281,6172.442970
min,0.000000,1.000000,4506.000000,16.000000,8305.000000,16.000000,7826.000000,16.000000,13.440649,144.808944,11.000000,1.000000,0.000000,0.000000,0.114961,1.000000,17.000000,1.000000,720.000000,14.000000,1.000000
25%,1.000000,2.000000,32459.250000,28.000000,47566.000000,25.000000,45549.000000,28.000000,34.152076,144.808944,11.000000,9.000000,0.615400,24244.000000,0.486853,1.000000,21.000000,1.000000,1099.000000,23.000000,725.000000
50%,1.000000,3.000000,44141.500000,47.000000,59562.500000,43.000000,57882.000000,47.000000,39.328977,144.808944,13.000000,13.000000,0.784100,37667.500000,0.629190,2.000000,22.000000,1.000000,1190.000000,25.000000,2264.000000
75%,1.000000,3.000000,61827.750000,98.000000,77911.750000,91.000000,75760.500000,97.000000,41.703058,145.037141,21.000000,15.000000,0.890100,58609.000000,0.777258,2.000000,25.000000,1.000000,1297.000000,28.000000,5711.000000
max,3.000000,99.000000,272682.000000,11263.000000,336392.000000,9665.000000,384547.000000,10468.000000,64.857560,145.721733,43.000000,18.000000,1.000000,179864.000000,0.996740,2.000000,48.000000,5.000000,1560.000000,35.000000,55120.000000


'Missing values per column:'

code                                  0
title                                 0
unit_id                               0
distance                              0
school_type                           0
credential_level                      0
1_yr_median_earnings               9924
1_yr_working_count                 9924
4_yr_median_earnings                  0
4_yr_working_count                    0
5_yr_median_earnings              14587
5_yr_working_count                14587
school_name                           0
school_state                          0
location_lat                          2
location_lon                      55918
locale                                0
carnegie_size_setting              4310
admission_rate_overall            18461
median_family_income                364
students_with_pell_grant           2356
open_admissions_policy             1019
age_entry                           364
title_iv_eligibility_type             0
sat_scores_average_overall        27506


'Correlation matrix:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
distance,1.000000,-0.064231,0.087875,0.045212,0.026595,0.060110,0.011334,0.049668,-0.023956,0.174078,0.027418,-0.029884,0.090494,-0.100275,0.084193,-0.063824,0.246856,0.149040,-0.125451,-0.139590,0.066216
credential_level,-0.064231,1.000000,0.528689,-0.049601,0.127210,-0.022065,0.542649,-0.059269,-0.005347,0.522233,-0.048364,0.127033,-0.021647,0.075802,-0.081029,0.117613,-0.045270,0.001576,0.025357,0.024428,0.092013
1_yr_median_earnings,0.087875,0.528689,1.000000,0.032196,0.912971,0.012966,0.871953,-0.004042,0.108403,-0.249537,-0.089905,0.216672,-0.185652,0.226013,-0.261235,0.237967,-0.088883,0.020126,0.243557,0.240116,0.163279
1_yr_working_count,0.045212,-0.049601,0.032196,1.000000,0.015639,0.972806,0.020124,0.826064,-0.054290,-0.302072,-0.056637,-0.041627,0.027674,-0.089292,0.078346,-0.092510,0.178741,0.072431,0.023925,0.051733,0.143269
4_yr_median_earnings,0.026595,0.127210,0.912971,0.015639,1.000000,0.013077,0.943206,-0.011333,0.135109,0.160781,-0.134777,0.336668,-0.277068,0.330410,-0.349991,0.335767,-0.202435,0.008376,0.365682,0.364367,0.217848
4_yr_working_count,0.060110,-0.022065,0.012966,0.972806,0.013077,1.000000,0.013833,0.884454,-0.051309,0.314415,-0.057614,-0.038005,0.030593,-0.087938,0.079297,-0.089302,0.177095,0.086418,0.016139,0.045802,0.117354
5_yr_median_earnings,0.011334,0.542649,0.871953,0.020124,0.943206,0.013833,1.000000,-0.014781,0.131825,-0.360540,-0.124917,0.342259,-0.284752,0.339342,-0.364824,0.354463,-0.223239,0.002439,0.374018,0.374147,0.209618
5_yr_working_count,0.049668,-0.059269,-0.004042,0.826064,-0.011333,0.884454,-0.014781,1.000000,-0.061832,0.953103,-0.061436,-0.071796,0.042269,-0.116831,0.112136,-0.122692,0.206762,0.059092,0.008662,0.059848,0.083087
location_lat,-0.023956,-0.005347,0.108403,-0.054290,0.135109,-0.051309,0.131825,-0.061832,1.000000,1.000000,0.109376,0.010372,0.108251,0.359543,-0.365666,0.094953,-0.113811,-0.023561,0.145722,0.151120,-0.041482
location_lon,0.174078,0.522233,-0.249537,-0.302072,0.160781,0.314415,-0.360540,0.953103,1.000000,1.000000,1.000000,1.000000,NaN,-1.000000,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN


In [13]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [14]:
model_df = df.copy()
target = '4_yr_median_earnings'

drop_columns=["title","4_yr_working_count","school_name",'1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count']
X = model_df.drop(columns=[target]).copy()
X = X.drop(columns=[],errors="ignore")
y = pd.to_numeric(model_df[target], errors='coerce').copy()
y_log = np.log10(y)

cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    "selectivity_bucket"
]
num_cols = [
    'admission_rate_overall',
    "location_lat",
    "location_lon", 
    'median_family_income',
    'students_with_pell_grant',
    'age_entry'
    
]

# categorical: force plain object and replace missing with np.nan
for c in cat_cols:
    X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})
    X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})

for c in cat_cols:
    X[c] = X[c].astype(object)
    X[c] = X[c].replace({pd.NA: np.nan})

# numeric: force numeric with np.nan
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# nuclear option: remove any lingering pd.NA anywhere in X
X = X.astype(object).replace({pd.NA: np.nan})

# now restore numeric cols back to numeric dtype
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

In [15]:
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log,
    test_size=0.2,
    random_state=RANDOM_STATE
)
# fix: coerce cat cols to string AFTER imputation
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        missing_values=np.nan, 
        strategy="constant", 
        fill_value="Missing"
    )),
    ("to_str", FunctionTransformer(
        lambda X: X.astype(str),  # ← force everything to string after imputing
        feature_names_out="one-to-one"  # ← add this
    )),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

In [16]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', Ridge())
])

param_grid = {
    'reg__alpha': [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid.fit(X_train, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV MAE:", round(-grid.best_score_, 2))
best_model = grid.best_estimator_
preds = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, preds), 2))
print("Test R2:", round(r2_score(y_test_log, preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.1}
Best CV MAE: 0.06
Test MAE: 0.06
Test R2: 0.7603


In [17]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

# Ridge uses coef_ not feature_importances_
coefficients = best_model.named_steps["reg"].coef_

importance = pd.Series(coefficients, index=feature_names)

# sort by absolute value to see most influential features
importance = importance.reindex(importance.abs().sort_values(ascending=False).index)

print("Most important features:")
print(importance.head(20).round(4).to_string())

Most important features:
cat__code_5105    0.3775
cat__code_4102    0.3259
cat__code_5133   -0.3127
cat__code_6001    0.3089
cat__code_5114    0.3046
cat__code_4103    0.2650
cat__code_5134   -0.2585
cat__code_5104    0.2510
cat__code_3014   -0.2475
cat__code_1509    0.2421
cat__code_4503   -0.2361
cat__code_1437    0.2335
cat__code_1425    0.2316
cat__code_4903    0.2261
cat__code_2805    0.2187
cat__code_5005   -0.2156
cat__code_5002   -0.2139
cat__code_1099   -0.2118
cat__code_1514    0.2109
cat__code_1409    0.2098


In [18]:
ls_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=100000, random_state=RANDOM_STATE))
])

ls_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

ls_grid = GridSearchCV(
    ls_pipe,
    param_grid=ls_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ls_grid.fit(X_train, y_train_log)

print("Best params:", ls_grid.best_params_)
print("Best CV MAE:", round(-ls_grid.best_score_, 2))

ls_best_model = ls_grid.best_estimator_
ls_preds = ls_best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, ls_preds), 2))
print("Test R2:", round(r2_score(y_test_log, ls_preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.01}
Best CV MAE: 0.12
Test MAE: 0.11
Test R2: 0.2675


In [19]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])


enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005, 0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_log)

print("ElasticNet Best params:", enet_grid.best_params_)
print("ElasticNet Best CV MAE:", round(-enet_grid.best_score_, 2))
enet_best_model = enet_grid.best_estimator_
enet_preds = enet_best_model.predict(X_test)

print("ElasticNet Test MAE:", round(mean_absolute_error(y_test_log, enet_preds), 2))
print("ElasticNet Test R2:", round(r2_score(y_test_log, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
ElasticNet Best params: {'reg__alpha': 0.01, 'reg__l1_ratio': 0.005}
ElasticNet Best CV MAE: 0.08
ElasticNet Test MAE: 0.08
ElasticNet Test R2: 0.625


In [20]:
baseline_pred = [y_train_log.mean()] * len(y_test_log)

print("Baseline MAE:", round(mean_absolute_error(y_test_log, baseline_pred), 2))

Baseline MAE: 0.13


In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict


# pipe_rfr = Pipeline([
#     ('preprocessor', preprocessor),
#     ('clf', RandomForestRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1
#     ))
# ])

# param_grid_rfr = {
#     'clf__n_estimators': [100, 200, 300],
#     'clf__max_depth': [None, 5, 10, 20],
#     'clf__min_samples_split': [2, 5, 10],
#     'clf__min_samples_leaf': [1, 2, 4]
# }

# grid_rfr = GridSearchCV(
#     estimator=pipe_rfr,
#     param_grid=param_grid_rfr,
#     scoring='neg_mean_absolute_error',
#     cv=cv,
#     n_jobs=1,
#     refit=True,
#     verbose=1,
#     error_score='raise'
# )

# grid_rfr.fit(X_train, y_train_log)

# print("RF Best params:", grid_rfr.best_params_)
# print("RF Best CV MAE:", round(-grid_rfr.best_score_, 2))

# best_rfr = grid_rfr.best_estimator_
# rfr_preds = best_rfr.predict(X_test)

# print("RF Test MAE:", round(mean_absolute_error(y_test_log, rfr_preds), 2))
# print("RF Test R2:", round(r2_score(y_test_log, rfr_preds), 4))

In [22]:
from xgboost import XGBRegressor

pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

param_grid_xgb = {
    'reg__n_estimators': [200, 300],
    'reg__max_depth': [5, 10], 
    'reg__learning_rate': [0.05, 0.1],
    'reg__subsample': [0.8, 1.0],
}

grid_xgb_log = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_xgb_log.fit(X_train, y_train_log)

print("XGB Best params:", grid_xgb_log.best_params_)
print("XGB Best CV MAE:", round(-grid_xgb_log.best_score_, 2))

best_xgb_log = grid_xgb_log.best_estimator_
xgb_preds_log = best_xgb_log.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE: 0.05
XGB Test MAE: 0.05
XGB Test R2: 0.8325


All models significantly outperformed the baseline. The linear model and Random Forest performed similarly, suggesting that linear relationships explain a large portion of the variance. However, XGBoost achieved the best performance, reducing MAE from 0.14 to 0.13 and increasing R2 to ~0.83. This indicates that nonlinear interactions exist in the data and are effectively captured by gradient boosting methods.

In [23]:
feature_names = best_xgb_log.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb_log.named_steps["reg"].feature_importances_

importance = pd.Series(importances, index=feature_names)

print("most important features")
display(importance.sort_values(ascending=False).head(15))
print("least important features")
display(importance.sort_values(ascending=True).head(15))

most important features


cat__code_1204                     0.081624
cat__open_admissions_policy_1.0    0.050494
cat__code_5138                     0.031034
cat__selectivity_bucket_elite      0.017238
cat__code_5007                     0.015058
cat__code_1107                     0.014384
cat__credential_level_7            0.014129
cat__code_1410                     0.013190
cat__code_5005                     0.010993
cat__code_1419                     0.010951
cat__code_1407                     0.010535
cat__code_1409                     0.010400
cat__code_5213                     0.010353
cat__code_1101                     0.010211
cat__code_5208                     0.009687
dtype: float32

least important features


cat__code_3029    0.0
cat__code_3032    0.0
cat__code_3044    0.0
cat__code_3199    0.0
cat__code_3601    0.0
cat__code_3701    0.0
cat__code_3800    0.0
cat__code_3899    0.0
cat__code_3905    0.0
cat__code_4001    0.0
cat__code_4002    0.0
cat__code_4004    0.0
cat__code_1307    0.0
cat__code_1306    0.0
cat__code_1302    0.0
dtype: float32

In [24]:
print("School level n:", X['unit_id'].nunique())
print('Program x Creds x Schools rows:', len(X))

School level n: 4967
Program x Creds x Schools rows: 55930


Feature importance analysis from the XGBoost model shows that categorical variables, particularly program codes (CIP), credential level, admission policy, and carnegie classification, were the most influential predictors. Additionally, several features had zero importance, indicating that certain categories did not contribute meaningfully to prediction, likely due to lack of signal.

The model was trained on log-transformed earnings to address skewness and improve predictive stability. Predictions were then exponentiated back to the original scale to evaluate performance and interpret errors in dollar terms.

Predictions are the typical (median-like) expected earnings rather than the average, which is appropriate given the skewed nature of income data.

In [25]:
def run_earnings_model(df, year, model_type="xgb", random_state=42, cv=5):
    year = int(year)
    target = f"{year}_yr_median_earnings"

    other_year_cols = []
    for y in [1, 4, 5]:
        if y != year:
            other_year_cols.extend([
                f"{y}_yr_median_earnings",
                f"{y}_yr_working_count",
            ])

    model_df = df.copy()
    model_df[target] = pd.to_numeric(model_df[target], errors="coerce")
    model_df = model_df[model_df[target].notna() & (model_df[target] > 0)].copy()

    drop_columns = ["title", "school_name"] + other_year_cols
    X = model_df.drop(columns=drop_columns, errors="ignore").copy()
    X = X.drop(columns=[target], errors="ignore")
    y = pd.to_numeric(model_df[target], errors="coerce").copy()
    y_log = np.log(y)

    cat_cols = [
        'code', 'school_type', 'locale', 'carnegie_size_setting',
        'open_admissions_policy', 'title_iv_eligibility_type',
        'credential_level', 'distance', 'selectivity_bucket', 'state'
    ]
    num_cols = [
        'admission_rate_overall', 'location_lat', 'location_lon',
        'median_family_income', 'students_with_pell_grant', 'age_entry'
    ]

    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in num_cols if c in X.columns]

    for c in cat_cols:
        X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X_train, X_test, y_train_log, y_test_log = train_test_split(
        X, y_log, test_size=0.2, random_state=random_state
    )

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("to_str", FunctionTransformer(lambda X: X.astype(str))),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])

    # ── model selection ──────────────────────────────────────────
    if model_type == "xgb":
        estimator = XGBRegressor(random_state=random_state, n_jobs=-1, verbosity=0)
        param_grid = {
            "reg__n_estimators": [200, 300],
            "reg__max_depth": [5, 10],
            "reg__learning_rate": [0.05, 0.1],
            "reg__subsample": [0.8, 1.0],
        }
    elif model_type == "ridge":
        estimator = Ridge()
        param_grid = {
            "reg__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
        }
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", estimator)
    ])

    grid = GridSearchCV(
        pipe, param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv, refit=True, verbose=1, error_score="raise"
    )

    grid.fit(X_train, y_train_log)

    print(f"\n===== {year}-YEAR MODEL ({model_type.upper()}) =====")
    print("Best params:", grid.best_params_)
    print("Best CV MAE (log):", round(-grid.best_score_, 4))

    best_model = grid.best_estimator_
    preds_log = best_model.predict(X_test)
    print("Test MAE (log):", round(mean_absolute_error(y_test_log, preds_log), 4))
    print("Test R² (log):", round(r2_score(y_test_log, preds_log), 4))

    # OOF predictions on full dataset
    cv_preds_log = cross_val_predict(
        best_model, X, y_log, cv=cv, method="predict", n_jobs=1
    )

    # build error_df
    actual_log_col = f"{year}_year_earning_log"
    pred_log_col   = f"{year}_year_pred_log"
    actual_col     = f"{year}_year_earning"
    pred_col       = f"{year}_year_pred"
    error_col      = f"{year}_year_error"

    error_df = X.copy()
    error_df[actual_log_col] = y_log.values
    error_df[pred_log_col]   = cv_preds_log

    cols_to_add = [c for c in ["title", "school_name", "credential_level"] if c in model_df.columns]
    error_df[cols_to_add] = model_df.loc[error_df.index, cols_to_add]

    error_df[actual_col] = np.exp(error_df[actual_log_col])
    error_df[pred_col]   = np.exp(error_df[pred_log_col])
    error_df[error_col]  = error_df[actual_col] - error_df[pred_col]

    return {
        "year": year,
        "model_type": model_type,
        "target": target,
        "grid": grid,
        "best_model": best_model,
        "X": X,
        "y": y,
        "error_df": error_df,
    }

In [26]:
results_1 = run_earnings_model(model_df, year=1,model_type="ridge", random_state=RANDOM_STATE, cv=cv)
results_4 = run_earnings_model(model_df, year=4,model_type="ridge", random_state=RANDOM_STATE, cv=cv)
results_5 = run_earnings_model(model_df, year=5,model_type="ridge", random_state=RANDOM_STATE, cv=cv)

Fitting 5 folds for each of 6 candidates, totalling 30 fits

===== 1-YEAR MODEL (RIDGE) =====
Best params: {'reg__alpha': 0.1}
Best CV MAE (log): 0.1627
Test MAE (log): 0.161
Test R² (log): 0.7705
Fitting 5 folds for each of 6 candidates, totalling 30 fits

===== 4-YEAR MODEL (RIDGE) =====
Best params: {'reg__alpha': 0.1}
Best CV MAE (log): 0.1429
Test MAE (log): 0.1426
Test R² (log): 0.7603
Fitting 5 folds for each of 6 candidates, totalling 30 fits

===== 5-YEAR MODEL (RIDGE) =====
Best params: {'reg__alpha': 1.0}
Best CV MAE (log): 0.1454
Test MAE (log): 0.1453
Test R² (log): 0.7774


In [27]:
df_1=results_1["error_df"]
display(df_1.head())
display(results_1["error_df"].shape)

df_4=results_4["error_df"]
display(df_4.head())
display(results_4["error_df"].shape)

df_5=results_5["error_df"]
display(df_5.head())
display(results_5["error_df"].shape)

,code,unit_id,distance,school_type,credential_level,1_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,1_year_earning_log,1_year_pred_log,title,school_name,1_year_earning,1_year_pred,1_year_error
1,1002,100654,1,Public,3,31.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.272911,10.029422,Audiovisual Communications Technologies/Techni...,Alabama A & M University,28938.0,22684.154521,6253.845479
2,1101,100654,1,Public,3,29.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.065075,10.833088,"Computer and Information Sciences, General.",Alabama A & M University,63900.0,50669.924596,13230.075404
3,1312,100654,2,Public,5,21.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.938361,10.781029,Teacher Education and Professional Development...,Alabama A & M University,56295.0,48099.587669,8195.412331
4,1410,100654,1,Public,3,40.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.187763,11.045013,"Electrical, Electronics, and Communications En...",Alabama A & M University,72241.0,62630.863733,9610.136267
5,1419,100654,1,Public,3,45.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.183782,10.941075,Mechanical Engineering.,Alabama A & M University,71954.0,56448.020039,15505.979961


(46006, 28)

,code,unit_id,distance,school_type,credential_level,4_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,4_year_earning_log,4_year_pred_log,title,school_name,4_year_earning,4_year_pred,4_year_error
0,0305,100654,1,Public,3,16,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.078274,10.866337,Forestry.,Alabama A & M University,64749.0,52382.997080,12366.002920
1,1002,100654,1,Public,3,37,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.651880,10.622161,Audiovisual Communications Technologies/Techni...,Alabama A & M University,42272.0,41034.213706,1237.786294
2,1101,100654,1,Public,3,39,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.390645,11.239211,"Computer and Information Sciences, General.",Alabama A & M University,88490.0,76054.897194,12435.102806
3,1312,100654,2,Public,5,24,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.008943,10.940271,Teacher Education and Professional Development...,Alabama A & M University,60412.0,56402.626078,4009.373922
4,1410,100654,1,Public,3,43,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.493182,11.372785,"Electrical, Electronics, and Communications En...",Alabama A & M University,98045.0,86923.569768,11121.430232


(55930, 28)

,code,unit_id,distance,school_type,credential_level,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error
2,1101,100654,1,Public,3,27.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.352968,11.276351,"Computer and Information Sciences, General.",Alabama A & M University,85218.0,78932.707587,6285.292413
3,1312,100654,2,Public,5,18.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.142760,10.894691,Teacher Education and Professional Development...,Alabama A & M University,69062.0,53889.526389,15172.473611
4,1410,100654,1,Public,3,29.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.412099,11.369727,"Electrical, Electronics, and Communications En...",Alabama A & M University,90409.0,86658.206509,3750.793491
5,1419,100654,1,Public,3,22.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.325740,11.248176,Mechanical Engineering.,Alabama A & M University,82929.0,76739.847717,6189.152283
8,2401,100654,1,Public,3,30.0,AL,34.783368,NaN,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.749935,10.795546,"Liberal Arts and Sciences, General Studies and...",Alabama A & M University,46627.0,48802.971591,-2175.971591


(41343, 28)

In [28]:
df_1['1_yr_working_count'].isna().sum()

0

In [29]:
merge_df=df_5.merge(df_4[['4_year_earning','4_year_earning_log','4_year_pred','4_year_pred_log','4_year_error',"4_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df=merge_df.merge(df_1[['1_year_earning','1_year_earning_log','1_year_pred','1_year_pred_log','1_year_error',"1_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df.shape

(36653, 40)

In [30]:
error_df=merge_df.copy()
valid_codes = (
    error_df.groupby(["code","credential_level"])["school_name"]
    .nunique()
)

valid_codes = valid_codes[valid_codes >= 3].index
valid_codes

MultiIndex([('0100', 3),
            ('0101', 2),
            ('0101', 3),
            ('0102', 2),
            ('0102', 3),
            ('0103', 2),
            ('0103', 3),
            ('0105', 3),
            ('0106', 2),
            ('0106', 3),
            ...
            ('5218', 3),
            ('5219', 2),
            ('5219', 3),
            ('5220', 2),
            ('5220', 3),
            ('5220', 5),
            ('5299', 3),
            ('5299', 5),
            ('5401', 3),
            ('5401', 5)],
           names=['code', 'credential_level'], length=590)

In [31]:
# error_df=error_df[error_df["4_yr_working_count"] >= 20]


error_df = (
    error_df
    .set_index(["code", "credential_level"])
    .loc[valid_codes]
    .reset_index()
)

error_df["school_count"] = (
    error_df.groupby(["code", "credential_level"])["school_name"]
    .transform("nunique")
)

error_df["confidence"] = pd.cut(
    error_df["school_count"],
    bins=[0, 5, 15, 100],
    labels=["low", "medium", "high"]
)

In [32]:
error_df.shape

(36219, 42)

In [33]:
years = ["1", "4", "5"]

for y in years:
    # percent error
    error_df[f"{y}_year_pct_error"] = (
        error_df[f"{y}_year_error"] / error_df[f"{y}_year_pred"]
    )

    # weight (you can keep using 4yr count OR match per year if you have it)
    k = np.percentile(np.log1p(error_df[f"{y}_yr_working_count"]), 75)

    weight = (
        np.log1p(error_df[f"{y}_yr_working_count"]) /
        np.log1p(error_df[f"{y}_yr_working_count"] + k)
    )

    # final score per year
    error_df[f"{y}_year_score"] = (
        error_df[f"{y}_year_pct_error"] * weight
    )

“The weighting scheme introduces only minor adjustments to ranking positions, suggesting that prediction error remains dominant while program size provides a secondary refinement.”

To avoid instability in groups with small sample sizes, a small constant (epsilon) was added to the standard deviation when computing the final score. This prevents artificially inflated scores caused by near-zero variance estimates, while still allowing all groups to be included in the analysis.

To account for differences in sample size, a soft penalization factor was applied using sqrt(n / (n + k)). This approach reduces the influence of groups with small sample sizes without excluding them entirely. As n increases, the penalty diminishes, allowing larger groups to retain their full weight while appropriately down-weighting less reliable estimates.

In [34]:
error_df["rank_1"] = error_df.groupby(["code","credential_level"])["1_year_score"].rank(ascending=False, method="min")
error_df["rank_4"] = error_df.groupby(["code","credential_level"])["4_year_score"].rank(ascending=False, method="min")
error_df["rank_5"] = error_df.groupby(["code","credential_level"])["5_year_score"].rank(ascending=False, method="min")

In [35]:
error_df["move_1_to_4"] = error_df["rank_4"] - error_df["rank_1"]
error_df["move_4_to_5"] = error_df["rank_5"] - error_df["rank_4"]
error_df["move_1_to_5"] = error_df["rank_5"] - error_df["rank_1"]

In [36]:
error_df["rank_std"] = error_df[["rank_1","rank_4","rank_5"]].std(axis=1)
group_size = error_df.groupby("code")["code"].transform("count")

error_df["rank_std_pct"] = error_df["rank_std"] / (group_size - 1)

In [37]:
error_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct
0,0100,3,110422,1,Public,21.0,CA,35.299513,NaN,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.027100,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,61518.918041,6831.081959,84412.0,11.343465,61849.328337,11.032457,22562.671663,36,64786.0,11.078845,44661.282885,10.706862,20124.717115,18.0,28,high,0.450608,0.418702,0.364801,0.352710,0.111040,0.104505,2.0,2.0,10.0,0.0,8.0,8.0,4.618802,0.171067
1,0100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.889413,"Agriculture, General.",Delaware State University,47478.0,53605.820876,-6127.820876,52676.0,10.871915,54423.909124,10.904559,-1747.909124,24,38873.0,10.568055,39038.464664,10.572303,-165.464664,22.0,28,high,-0.004239,-0.003998,-0.032117,-0.030432,-0.114313,-0.108515,16.0,15.0,18.0,-1.0,3.0,2.0,1.527525,0.056575
2,0100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.967096,"Agriculture, General.",Illinois State University,64041.0,57936.128323,6104.871677,63600.0,11.060369,59188.291176,10.988479,4411.708824,214,47295.0,10.764160,44425.661994,10.701573,2869.338006,205.0,28,high,0.064587,0.064311,0.074537,0.074227,0.105372,0.104631,11.0,8.0,9.0,-3.0,1.0,-2.0,1.527525,0.056575
3,0100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.804913,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,49262.255760,13768.744240,57596.0,10.961208,50034.114619,10.820460,7561.885381,47,39700.0,10.589106,38819.886275,10.566688,880.113725,22.0,28,high,0.022672,0.021384,0.151135,0.147450,0.279499,0.264632,13.0,4.0,2.0,-9.0,-2.0,-11.0,5.859465,0.217017
4,0100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.873760,"Agriculture, General.",Western Illinois University,58204.0,52773.247380,5430.752620,58333.0,10.973923,52459.403970,10.867795,5873.596030,160,48509.0,10.789505,38882.312529,10.568295,9626.687471,149.0,28,high,0.247585,0.246045,0.111965,0.111311,0.102907,0.102258,4.0,5.0,11.0,1.0,6.0,7.0,3.785939,0.140220


In [38]:
error_df.shape

(36219, 56)

In [39]:
error_df["rank_std_pct"].describe()

count    36219.000000
mean         0.093440
std          0.090882
min          0.000000
25%          0.025381
50%          0.063758
75%          0.133090
max          0.577350
Name: rank_std_pct, dtype: float64

When ranking schools within the same program, we observe an average rank movement of about 11%, indicating that performance is not stable over time even within comparable fields.

In [40]:
import plotly.express as px

px.histogram(error_df["rank_std_pct"])

In [41]:
error_df[["rank_std_pct", "4_yr_working_count"]].corr()

,rank_std_pct,4_yr_working_count
rank_std_pct,1.000000,-0.111136
4_yr_working_count,-0.111136,1.000000


In [42]:
top = error_df.nsmallest(100, "rank_4")   # best schools
bottom = error_df.nlargest(100, "rank_4")  # worst schools

print('top 100 std:',top["rank_std_pct"].mean())
print('bottom 100 std:',bottom["rank_std_pct"].mean())

top 100 std: 0.06521549606814324
bottom 100 std: 0.05979689955509088


We tested whether ranking instability was due to small sample sizes, but found no meaningful relationship. Even top-performing schools show similar or increased volatility, suggesting the instability is inherent to the earnings metric itself rather than noise.

In [43]:
program_stability = (
    error_df
    .groupby(["code", "credential_level"])
    .agg(
        mean_rank_std_pct=("rank_std_pct", "mean"),
        n=("school_name", "nunique")
    )
    .reset_index()
)

stable_programs = program_stability[
    program_stability["mean_rank_std_pct"] < 2   #Used to filter for stable programs
]
stable_df = error_df.merge(
    stable_programs[["code", "credential_level",'mean_rank_std_pct']],
    on=["code", "credential_level"],
    how="inner"
)
len(stable_df) / len(error_df)

1.0

In [44]:
stable_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct
0,0100,3,110422,1,Public,21.0,CA,35.299513,NaN,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.027100,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,61518.918041,6831.081959,84412.0,11.343465,61849.328337,11.032457,22562.671663,36,64786.0,11.078845,44661.282885,10.706862,20124.717115,18.0,28,high,0.450608,0.418702,0.364801,0.352710,0.111040,0.104505,2.0,2.0,10.0,0.0,8.0,8.0,4.618802,0.171067,0.16765
1,0100,3,130934,1,Public,24.0,DE,39.187173,NaN,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.889413,"Agriculture, General.",Delaware State University,47478.0,53605.820876,-6127.820876,52676.0,10.871915,54423.909124,10.904559,-1747.909124,24,38873.0,10.568055,39038.464664,10.572303,-165.464664,22.0,28,high,-0.004239,-0.003998,-0.032117,-0.030432,-0.114313,-0.108515,16.0,15.0,18.0,-1.0,3.0,2.0,1.527525,0.056575,0.16765
2,0100,3,145813,1,Public,132.0,IL,40.509403,NaN,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.967096,"Agriculture, General.",Illinois State University,64041.0,57936.128323,6104.871677,63600.0,11.060369,59188.291176,10.988479,4411.708824,214,47295.0,10.764160,44425.661994,10.701573,2869.338006,205.0,28,high,0.064587,0.064311,0.074537,0.074227,0.105372,0.104631,11.0,8.0,9.0,-3.0,1.0,-2.0,1.527525,0.056575,0.16765
3,0100,3,149222,1,Public,23.0,IL,37.714193,NaN,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.804913,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,49262.255760,13768.744240,57596.0,10.961208,50034.114619,10.820460,7561.885381,47,39700.0,10.589106,38819.886275,10.566688,880.113725,22.0,28,high,0.022672,0.021384,0.151135,0.147450,0.279499,0.264632,13.0,4.0,2.0,-9.0,-2.0,-11.0,5.859465,0.217017,0.16765
4,0100,3,149772,1,Public,145.0,IL,40.468086,NaN,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.873760,"Agriculture, General.",Western Illinois University,58204.0,52773.247380,5430.752620,58333.0,10.973923,52459.403970,10.867795,5873.596030,160,48509.0,10.789505,38882.312529,10.568295,9626.687471,149.0,28,high,0.247585,0.246045,0.111965,0.111311,0.102907,0.102258,4.0,5.0,11.0,1.0,6.0,7.0,3.785939,0.140220,0.16765


In [45]:
stable_programs.shape

(590, 4)

In [46]:
stable_gap_df = stable_df.copy()

stable_gap_df["median_score"] = stable_gap_df[
    ["1_year_score", "4_year_score", "5_year_score"]
].median(axis=1)

# program-level gap stats
program_gap = (
    stable_gap_df
    .groupby(["code", "credential_level"])
    .agg(
        min_score=("median_score", "min"),
        q1_score=("median_score", lambda x: x.quantile(0.25)),
        median_program_score=("median_score", "median"),
        q3_score=("median_score", lambda x: x.quantile(0.75)),
        max_score=("median_score", "max"),
        mean_score=("median_score", "mean"),
        std_score=("median_score", "std"),
        n=("school_name", "nunique")
    )
    .reset_index()
)

# raw gap: biggest under/over performer spread 
program_gap["gap"] = program_gap["max_score"] - program_gap["min_score"]

# robust gap: less sensitive to one weird school 
program_gap["iqr_gap"] = program_gap["q3_score"] - program_gap["q1_score"]

#  weight so tiny groups don't dominate 
program_gap["gap_weighted"] = (
    program_gap["gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)
program_gap["iqr_gap_weighted"] = (
    program_gap["iqr_gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)

#  bring back titles 
titles = error_df[["code", "credential_level", "title"]].drop_duplicates()

program_gap = program_gap.merge(
    titles,
    on=["code", "credential_level"],
    how="left"
)

# top programs with biggest stable-school spread 
top_gap_programs = (
    program_gap
    .sort_values("gap_weighted", ascending=False)
)

top_iqr_programs = (
    program_gap
    .sort_values("iqr_gap_weighted", ascending=False)
)

# views
top_gap_programs.head(20)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
303,3099,3,-0.309400,-0.129214,-0.037518,0.077818,1.984911,0.002353,0.243483,145,2.294311,0.207031,2.146291,0.193675,"Multi/Interdisciplinary Studies, Other."
229,2201,7,-0.417765,-0.149786,-0.052471,0.133293,1.788570,0.060040,0.369424,164,2.206335,0.283078,2.079534,0.266810,Law.
536,5202,5,-0.365262,-0.026973,0.099641,0.260931,1.507207,0.152200,0.268808,703,1.872469,0.287905,1.846207,0.283867,"Business Administration, Management and Operat..."
480,5112,7,-0.490506,-0.104476,0.004112,0.125330,1.474785,0.046234,0.254950,123,1.965291,0.229805,1.817525,0.212527,Medicine.
470,5109,2,-0.651611,-0.071265,0.023269,0.141557,1.059330,0.053918,0.183285,417,1.710942,0.212822,1.670873,0.207838,"Allied Health Diagnostic, Intervention, and Tr..."
516,5138,2,-0.607322,-0.004596,0.083549,0.178407,1.061767,0.101971,0.180066,815,1.669090,0.183003,1.648858,0.180785,"Registered Nursing, Nursing Administration, Nu..."
346,4301,1,-0.372968,0.169335,0.455652,0.706198,1.424726,0.474148,0.386257,97,1.797693,0.536862,1.629684,0.486688,Criminal Justice and Corrections.
65,1101,3,-0.422795,-0.068175,0.048728,0.181640,1.225077,0.078881,0.223481,351,1.647872,0.249815,1.602224,0.242894,"Computer and Information Sciences, General."
464,5107,5,-0.219391,0.029737,0.142161,0.278900,1.473257,0.182495,0.247376,153,1.692648,0.249164,1.588804,0.233877,Health and Medical Administrative Services.
469,5109,1,-0.581478,-0.124522,0.015237,0.185366,1.072883,0.056020,0.263258,155,1.654362,0.309888,1.554097,0.291107,"Allied Health Diagnostic, Intervention, and Tr..."


In [47]:
top_iqr_programs.head(10)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
468,5108,5,0.454975,0.750102,1.538336,2.488454,3.035115,1.661195,0.991847,9,2.580140,1.738351,1.222171,0.823429,Allied Health and Medical Assisting Services.
423,5004,1,-0.231893,0.104245,1.328822,1.675248,2.217767,1.013170,0.957553,8,2.449660,1.571003,1.088738,0.698223,Design and Applied Arts.
528,5199,5,-0.318943,-0.260769,-0.111036,0.718182,1.300994,0.201420,0.613677,10,1.619937,0.978951,0.809969,0.489475,Health Professions and Related Clinical Scienc...
346,4301,1,-0.372968,0.169335,0.455652,0.706198,1.424726,0.474148,0.386257,97,1.797693,0.536862,1.629684,0.486688,Criminal Justice and Corrections.
459,5106,2,-0.523692,0.126362,0.285399,0.479694,1.078059,0.289254,0.300213,165,1.601751,0.353332,1.510223,0.333141,Dental Support Services and Allied Professions.
313,3802,3,-0.426326,-0.309429,0.025906,0.131457,0.449459,-0.027359,0.272087,22,0.875784,0.440886,0.602102,0.303109,Religion/Religious Studies.
239,2313,3,-0.304416,-0.121633,0.004834,0.209200,0.431807,0.041804,0.190535,82,0.736223,0.330833,0.656199,0.294873,Rhetoric and Composition/Writing Studies.
469,5109,1,-0.581478,-0.124522,0.015237,0.185366,1.072883,0.056020,0.263258,155,1.654362,0.309888,1.554097,0.291107,"Allied Health Diagnostic, Intervention, and Tr..."
551,5207,5,-0.335192,-0.075065,0.365763,0.939225,1.597259,0.498398,0.854079,4,1.932451,1.014290,0.552129,0.289797,Entrepreneurial and Small Business Operations.
536,5202,5,-0.365262,-0.026973,0.099641,0.260931,1.507207,0.152200,0.268808,703,1.872469,0.287905,1.846207,0.283867,"Business Administration, Management and Operat..."


In [48]:
print(
    program_stability[
        (program_stability["code"] == "4301") &
        (program_stability["credential_level"] == 1)
    ]
)

     code  credential_level  mean_rank_std_pct   n
346  4301                 1           0.007756  97


In [49]:
(
    stable_gap_df[(stable_gap_df["code"] == "4301")&(stable_gap_df["credential_level"]==1)]
    .sort_values(["credential_level", "median_score"], ascending=[True, False])
)

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
16289,4301,1,123013,1,Public,105.0,CA,38.457090,NaN,12,4.0,NaN,19251.0,0.977082,1.0,24.0,1,NaN,NaN,NaN,Missing,11.444689,10.582769,Criminal Justice and Corrections.,Santa Rosa Junior College,93404.0,39449.180591,53954.819409,106298.0,11.574002,43478.134351,10.680013,62819.865649,76,111649.0,11.623115,30346.949345,10.320451,81302.050655,54.0,97,high,2.679085,2.624514,1.444861,1.424726,1.367704,1.355120,1.0,1.0,1.0,0.0,0.0,0.0,0.000000,0.000000,0.007756,1.424726
16290,4301,1,123527,1,Public,19.0,CA,34.086353,NaN,12,4.0,NaN,16948.0,NaN,1.0,25.0,1,NaN,NaN,NaN,Missing,10.724588,10.558701,Criminal Justice and Corrections.,San Bernardino Valley College,45460.0,38511.085820,6948.914180,99385.0,11.506756,41128.860398,10.624465,58256.139602,25,94285.0,11.454077,30109.031032,10.312580,64175.968968,45.0,97,high,2.131452,2.077810,1.416430,1.345460,0.180439,0.168581,2.0,2.0,71.0,0.0,69.0,69.0,39.837169,0.039018,0.007756,1.345460
16359,4301,1,226134,1,Public,50.0,TX,27.506477,NaN,11,12.0,NaN,17884.0,NaN,1.0,21.0,1,NaN,NaN,NaN,Missing,11.282871,10.430743,Criminal Justice and Corrections.,Laredo College,79449.0,33885.507288,45563.492712,77575.0,11.259000,36903.492476,10.516061,40671.507524,43,72049.0,11.185102,26158.292208,10.171922,45890.707792,25.0,97,high,1.754346,1.667961,1.102105,1.072328,1.344631,1.315195,3.0,4.0,2.0,1.0,-2.0,-1.0,1.000000,0.000979,0.007756,1.315195
16374,4301,1,441760,2,Public,24.0,TX,30.047885,NaN,12,3.0,NaN,24907.0,0.812036,1.0,24.0,1,NaN,NaN,NaN,Missing,11.315462,10.484848,Criminal Justice and Corrections.,Lamar Institute of Technology,82081.0,35769.408929,46311.591071,76188.0,11.240959,39867.862647,10.593326,36320.137353,31,71733.0,11.180706,29321.428989,10.286074,42411.571011,26.0,97,high,1.446436,1.378283,0.911013,0.875203,1.294726,1.229059,6.0,12.0,3.0,6.0,-9.0,-3.0,4.582576,0.004488,0.007756,1.229059
16312,4301,1,136358,1,Public,139.0,FL,26.612560,NaN,21,15.0,NaN,21408.0,0.921804,1.0,23.0,1,NaN,NaN,NaN,Missing,11.288531,10.498203,Criminal Justice and Corrections.,Palm Beach State College,79900.0,36250.316313,43649.683687,78216.0,11.267230,40579.477284,10.611018,37636.522716,153,69415.0,11.147858,27529.570338,10.223016,41885.429662,168.0,97,high,1.521471,1.513244,0.927477,0.921773,1.204119,1.196141,4.0,11.0,4.0,7.0,-7.0,0.0,4.041452,0.003958,0.007756,1.196141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16335,4301,1,186034,2,Public,58.0,NJ,40.918449,NaN,21,3.0,NaN,18422.0,0.938905,1.0,24.0,1,NaN,NaN,NaN,Missing,10.673110,10.581535,Criminal Justice and Corrections.,Passaic County Community College,43179.0,39400.528447,3778.471553,38554.0,10.559815,44643.416630,10.706462,-6089.416630,122,23559.0,10.067263,31368.643061,10.353564,-7809.643061,20.0,97,high,-0.248963,-0.233254,-0.136401,-0.135308,0.095899,0.094133,96.0,93.0,75.0,-3.0,-18.0,-21.0,11.357817,0.011124,0.007756,-0.135308
16380,4301,1,482291,1,"Private, for-profit",102.0,CA,35.381617,NaN,11,Missing,NaN,6719.0,0.951503,1.0,26

In [50]:
import plotly.express as px
bar_df = stable_gap_df[
    (stable_gap_df["code"] == "4301") & (stable_gap_df["credential_level"] == 1)
]

bar_df = bar_df.sort_values(by="median_score", ascending=True)

px.bar(
    bar_df,
    x="median_score",
    y="school_name",
    orientation="h",
    labels={"median_score":"3 year Pred. Error Average","school_name":"Schools"},
    title="Certificate in Criminal Justice and Corrections."
)

Programs such as Criminal Justice show substantial variability in outcomes across institutions, even after controlling for observable factors. This suggests that institutional effects such as program quality, networking opportunities, or industry connections may play a significant role in shaping student outcomes.

In [51]:
save(stable_gap_df,file_name="national_residual_programs")